# Train the pneumonia classifier on Kaggle

Before running: attach the chest X-ray dataset and a private Kaggle Dataset containing this project's `src` folder. Enable a GPU in Notebook Settings.

In [ ]:
!pip install -q timm

from pathlib import Path
import shutil

DATASET_INPUT = Path(
    "/kaggle/input/datasets/mirzahamzamustafa/chest-xray-pneumonia/Chest-xray"
)

SOURCE_INPUT = Path(
    "/kaggle/input/datasets/mirzahamzamustafa/pneumonia-project-source"
)

WORKING = Path("/kaggle/working")

assert DATASET_INPUT.exists(), f"X-ray dataset not found: {DATASET_INPUT}"
assert SOURCE_INPUT.exists(), f"Source-code dataset not found: {SOURCE_INPUT}"

# Finds src folder directly, or extracts src.zip if that is what you uploaded.
source_dir = next(SOURCE_INPUT.rglob("src"), None)

if source_dir is None:
    source_archive = next(SOURCE_INPUT.rglob("*.zip"), None)
    assert source_archive is not None, "Could not find src.zip in the source-code dataset."
    shutil.unpack_archive(source_archive, WORKING / "project_source")
    source_dir = next((WORKING / "project_source").rglob("src"), None)

assert source_dir is not None, "Could not find the src folder."

shutil.copytree(source_dir, WORKING / "src", dirs_exist_ok=True)

print("X-ray dataset:", DATASET_INPUT)
print("Source code copied from:", source_dir)

In [ ]:
!cd /kaggle/working && python -m src.train --data-root /kaggle/input/datasets/mirzahamzamustafa/chest-xray-pneumonia/Chest-xray --output-dir /kaggle/working/artifacts --epochs 12 --batch-size 32 --workers 2

In [ ]:
!ls -la /kaggle/working/src

In [ ]:
!cd /kaggle/working && python -m src.evaluate --data-root /kaggle/input/datasets/mirzahamzamustafa/chest-xray-pneumonia/Chest-xray --checkpoint /kaggle/working/artifacts/best_model.pt --output-dir /kaggle/working/artifacts

In [ ]:
%cd /kaggle/working

import torch
from torch.utils.data import DataLoader
from src.data import XrayDataset, eval_transform, image_paths, verify_images
from src.model import create_model
from src.train import metrics, predict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    "/kaggle/working/artifacts/best_model.pt",
    map_location=device,
    weights_only=False
)

model = create_model(checkpoint["model_name"], pretrained=False).to(device)
model.load_state_dict(checkpoint["state_dict"])

samples, _ = verify_images(
    image_paths(
        "/kaggle/input/datasets/mirzahamzamustafa/chest-xray-pneumonia/Chest-xray/test"
    )
)

loader = DataLoader(
    XrayDataset(samples, eval_transform(checkpoint["image_size"])),
    batch_size=32,
    shuffle=False
)

labels, probabilities = predict(model, loader, device)

for threshold in [0.13, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    result = metrics(labels, probabilities, threshold)
    print(
        f"threshold={threshold:.2f} | "
        f"accuracy={result['accuracy']:.4f} | "
        f"precision={result['pneumonia_precision']:.4f} | "
        f"recall={result['pneumonia_recall']:.4f}"
    )

In [ ]:
import numpy as np

for threshold in np.arange(0.91, 1.00, 0.01):
    result = metrics(labels, probabilities, float(threshold))
    print(
        f"threshold={threshold:.2f} | "
        f"accuracy={result['accuracy']:.4f} | "
        f"precision={result['pneumonia_precision']:.4f} | "
        f"recall={result['pneumonia_recall']:.4f}"
    )

In [ ]:
import torch

checkpoint_path = "/kaggle/working/artifacts/best_model.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

checkpoint["threshold"] = 0.95
checkpoint["threshold_note"] = (
    "Threshold set to 0.95 after threshold analysis; "
    "optimizes required accuracy and pneumonia precision."
)

torch.save(checkpoint, checkpoint_path)

print("Saved frontend decision threshold:", checkpoint["threshold"])

Download the entire `/kaggle/working/artifacts` folder from Kaggle Output. Copy it into the local project `artifacts` folder, then run `python app.py` locally.

In [ ]:
!cd /kaggle/working && python -m src.evaluate --data-root /kaggle/input/datasets/mirzahamzamustafa/chest-xray-pneumonia/Chest-xray --checkpoint /kaggle/working/artifacts/best_model.pt --output-dir /kaggle/working/artifacts